# 18 — CrewAI: the team archetype (and a lesson in dependency pins)

**What you'll learn**

- That a CrewAI `Crew` is a team of role-playing `Agent`s working through `Task`s under a `Process` — the multi-agent shapes you hand-built in ch07 (specialists, a supervisor, handoffs), now packaged as an org chart
- Why this chapter needs its **own virtual environment and kernel** — a real, resolver-proven dependency clash (`crewai` pins `mcp~=1.28.1` and `pydantic<2.13`, both *below* what the main stack resolved), not a footnote
- The model wiring: a CrewAI `LLM` is a LiteLLM router string, so the same `openrouter/deepseek` you have used since ch01 drops straight in
- A two-role crew — a `triager` that gathers facts and a `policy checker` that verifies them against store policy — collaborating over one Larkspur ticket via `Process.sequential`
- The honest read: when the team metaphor buys real isolation and when it is ch07's tax — an extra agent, an extra boundary, an org chart that feels tidy but costs tokens

*Time: ~2 min. Cost: ~$0.01 (one small two-agent crew; the crew's LiteLLM calls share the disk cache, so reruns are ~free).*

> **Before running this notebook** — CrewAI installs into its **own** virtual environment with its **own** Jupyter kernel. It cannot share the main course venv; the next section proves why. Set it up once:
>
> ```
> python -m venv .venv-crewai && . .venv-crewai/bin/activate
> pip install -e ".[crewai]"
> python -m ipykernel install --user --name agentic-lab-crewai
> ```
>
> Then, in Jupyter, pick the kernel **agentic-lab-crewai** for THIS notebook (Kernel -> Change Kernel). Every cell below runs in that venv — `crewai`, plus the same `shoplab`, `litellm`, and `diskcache` you already know — not the main one.

## A crew is an org chart drawn in prompts

Chapter 07 built multi-agent systems from parts you could read line by line. A **specialist** was `run_agent` given a scoped toolset and a role-specific system prompt. A **supervisor** called that specialist like any other tool and read back a result. A **handoff** forwarded the running conversation to a differently-instructed agent that finished the job. And the thesis under all of it was a warning: *more agents is a cost, not a default* — every agent you add is another context to fill and another boundary a finding degrades crossing.

[CrewAI](https://docs.crewai.com/) is that same picture, packaged as an org chart. Its docs describe it as a way to "build collaborative AI agents, crews, and flows." An `Agent` carries a `role`, a `goal`, and a `backstory` — your ch07 specialist's system prompt, split into three labelled fields. A `Task` is a unit of work assigned to an agent — the question you handed a specialist. A `Crew` bundles agents and tasks and runs them under a `Process`: `sequential` (one after another, each task's output feeding the next) or `hierarchical` (a manager agent delegates). Nothing here is a new idea about agents. It is the ch07 team, given names, defaults, and a runtime — and, as we will see, its own dependency tree.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

The frozen tracing cell stays in place for consistency, but note the seam: this notebook runs in the isolated `.venv-crewai`, which deliberately does **not** install `arize-phoenix` (that would drag the very dependencies we are isolating from back in). So `obs.enable_phoenix()` detects the missing package and skips cleanly — CrewAI's own calls still run, just untraced here. To watch a crew in Phoenix, trace it from the main venv, or add `openinference-instrumentation-crewai` to this one.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## A lesson in dependency pins

Here is the real reason for the separate venv, and it is worth more than a footnote. A framework does not just bring its own API — it brings its own **dependency tree**, and that tree has opinions that can contradict the rest of your stack.

The main course stack pins `mcp>=1.28,<2` on purpose: it teaches the 1.x `FastMCP` server (ch13), and [the MCP Python SDK README](https://github.com/modelcontextprotocol/python-sdk) warns that "since `pip install mcp` now installs 2.x, keep a `<2` upper bound ... (for example `mcp>=1.28,<2`) until you've migrated." Given that band, the main venv resolved `mcp` to **1.29.0** and `pydantic` to **2.13.4** — the newest releases still under the caps.

CrewAI's tree is stricter. It requires `mcp~=1.28.1` (that is: `>=1.28.1, <1.29`) and `pydantic<2.13` — *both below* what the main stack resolved. There is no single set of versions that satisfies both projects, so `pip` cannot install `crewai` into the main venv without downgrading `mcp` and `pydantic` and risking every notebook that depends on them. A fresh venv is not tidiness; it is the only correct resolution. The next cell proves the clash from inside this very environment.

In [ ]:
import importlib.metadata as im
from packaging.requirements import Requirement

# What the MAIN course stack (.venv, `pip install -e ".[obs]"`) actually resolved:
MAIN_STACK = {"mcp": "1.29.0", "pydantic": "2.13.4", "openai": "2.54.0"}

crewai_pins = {Requirement(r).name: Requirement(r).specifier
               for r in im.requires("crewai")
               if Requirement(r).name in MAIN_STACK}

for name, main_ver in MAIN_STACK.items():
    spec = crewai_pins[name]
    ok = spec.contains(main_ver, prereleases=True)
    print(f"{name:9} crewai pins {str(spec):16} | .venv-crewai has {im.version(name):8} "
          f"| main stack {main_ver} -> {'ok' if ok else 'CONFLICT'}")

> **What you should see:** two `CONFLICT` lines and one `ok`. `mcp` (`~=1.28.1`) and `pydantic` (`<2.13`) both refuse the versions the main stack resolved (`1.29.0`, `2.13.4`) — that is the pair that forces a separate venv. `openai` (`<3`) happens to be satisfied by the main stack today, but it is the pin that keeps CrewAI and the OpenAI Agents SDK (which needs `openai>=3`) from ever sharing one environment. The lesson generalizes: before you adopt a framework, resolve its pins against your stack — a beautiful API is worth nothing if it cannot coexist with the code you already run.

## The model: a CrewAI `LLM` is a LiteLLM string

CrewAI calls models through LiteLLM under the hood — the same router you met at the ch01 model boundary. So a CrewAI `LLM` takes the exact `openrouter/deepseek/deepseek-v3.2` string the whole course uses; it parses that into a **provider** (`openrouter`) and a **model**, points itself at OpenRouter's endpoint, and reads `OPENROUTER_API_KEY` from the environment. We reuse the frozen `MODEL` and `TEMPERATURE` rather than hard-coding anything. (Two environment flags quiet CrewAI's telemetry and OpenTelemetry exporter, which otherwise phone home and add noise.)

In [ ]:
os.environ["CREWAI_TELEMETRY_OPT_OUT"] = "true"   # do not phone telemetry.crewai.com
os.environ["OTEL_SDK_DISABLED"] = "true"
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from shoplab import world

orders = {o["order_id"]: o for o in world.load_orders()}
customers = {c["customer_id"]: c for c in world.load_customers()}

# CrewAI -> LiteLLM: the same "openrouter/..." routing string as every other chapter.
llm = LLM(model=MODEL, temperature=TEMPERATURE)
print("crewai LLM -> provider:", llm.provider, "| model:", llm.model)

## Larkspur tools, wrapped as CrewAI tools

An agent needs the same ops-desk lookups it always uses. CrewAI's `@tool("name")` decorator turns a docstringed function into a tool the agent can call — the docstring becomes the description, the signature becomes the schema. This is exactly what `shoplab.tools.Tool` did by hand in ch02; here the decorator wires it. We expose three read-only lookups — no money-moving tools, because these two agents only need to *decide*, not to execute (least privilege, ch07's wall).

In [ ]:
@tool("get_order")
def get_order(order_id: str) -> dict:
    """Look up a Larkspur order by id (items, totals, status, dates)."""
    return orders.get(order_id, {"error": f"no such order {order_id}"})

@tool("get_customer")
def get_customer(customer_id: str) -> dict:
    """Look up a Larkspur customer by id (tier, flags, history)."""
    return customers.get(customer_id, {"error": f"no such customer {customer_id}"})

@tool("find_policy")
def find_policy(query: str) -> list:
    """Keyword-search the 12 Larkspur store policy documents (returns the top 2)."""
    return world.search_policy(query, k=2)

print("crewai tools:", [get_order.name, get_customer.name, find_policy.name])

## A two-role crew: triager and policy checker

Now the team. Two agents, each a ch07 specialist by another name. The **triager** holds `get_order` and `get_customer` and gathers the facts, proposing a decision and amount. The **policy checker** holds only `find_policy`; its job is to take the triager's proposal and verify it against the governing store policy — catching, for instance, a wrong restocking fee. Splitting the toolset this way is the legitimate reason ch07 gives for a second agent: *isolation and a different toolset*, not an org chart for its own sake.

In [ ]:
triager = Agent(
    role="Larkspur returns triager",
    goal="Gather the order and customer facts for a return ticket and propose a decision.",
    backstory="You work the Larkspur ops desk and know every order and customer by heart.",
    tools=[get_order, get_customer],
    llm=llm, allow_delegation=False, max_iter=6, verbose=False)

checker = Agent(
    role="Larkspur policy checker",
    goal="Verify a proposed return decision against the governing store policy and finalize it.",
    backstory="You are the policy desk: you cite the exact policy that rules each return.",
    tools=[find_policy],
    llm=llm, allow_delegation=False, max_iter=6, verbose=False)

print("roles:", triager.role, "|", checker.role)

### Tasks, and the handoff between them

A `Task` is a unit of work with a `description`, an `expected_output`, and an `agent`. The interesting wiring is `context=[triage_task]` on the second task: it forwards the first task's output to the checker — CrewAI's version of ch07's `forward()`, the message list passed from one agent to the next. Bundle both into a `Crew` with `Process.sequential` and the crew runs task one, hands its result to task two, and returns the last output.

One Jupyter wrinkle worth naming: in a plain script you would call `crew.kickoff()`, but a notebook kernel already runs an asyncio event loop, and CrewAI refuses to block it — it asks for the async entrypoint `crew.kickoff_async()`, which a top-level `await` runs directly.

In [ ]:
TICKET = ("Ticket TKT-2205: order ORD-7312, customer CUST-07, sku LK-1016, qty 1. "
          "item_condition=opened, days_since_delivery=18 (authoritative — do NOT recompute "
          "from dates), requested_action=refund, evidence_photo=false. The customer opened the "
          "Torrent boots, wore them one evening indoors, they pinch at the toes; repacked with tags.")

triage_task = Task(
    description="Triage this return. " + TICKET + " Use get_order and get_customer to look up "
        "the facts. State the item value, whether the customer is vip, and propose a decision "
        "(approve_refund/partial_refund/replacement/store_credit/deny/escalate) with the amount.",
    expected_output="A short proposal: item value, customer tier, proposed decision, and amount.",
    agent=triager)

check_task = Task(
    description="Take the triager's proposal and check it against store policy. Use find_policy "
        "for an opened, in-window return by a non-vip customer. Confirm or correct decision and amount.",
    expected_output="Exactly one line: '<decision> | <policy_id> | $<amount>'.",
    agent=checker, context=[triage_task])

crew = Crew(agents=[triager, checker], tasks=[triage_task, check_task],
            process=Process.sequential, verbose=False)
print("crew:", type(crew).__module__ + "." + type(crew).__name__, "| process:", crew.process)

In [ ]:
result = await crew.kickoff_async()   # async: a notebook already runs an event loop

for step in result.tasks_output:
    print(f"[{step.agent}]\n{str(step.raw).strip()}\n")
print("gold (TKT-2205): partial_refund | pol-restocking | $170.99")
print("token usage:", result.token_usage.total_tokens, "tokens over",
      result.token_usage.successful_requests, "requests")

> **What you should see:** the collaboration, out loud. The triager reads the order and customer and proposes a refund on the $189.99 boot — but, holding no policy tool, it *guesses* the fee, and its proposal wobbles run to run (a full $189.99, or 90%, or some invented member perk). The policy checker then searches, lands on `pol-restocking`, applies the real 10% fee, and computes the refund at **$170.99** — the gold amount and policy id (it may round to $171). Watch the one field that still drifts: the decision *label* comes back `approve_refund` or `partial_refund` depending on the run — the exact boundary wobble ch07 measured, since both names describe "refund 90%" and gold is `partial_refund`. Two roles, one ticket: the second agent earned its keep because it held a tool the first did not, and fixed the number that actually moves money. That is ch07's rule, playing out inside a `Crew`.

## The crew is your ch07 team, renamed

Line the vocabulary up and CrewAI stops being magic. Every concept in the run above is a ch07 shape wearing a CrewAI label.

| CrewAI concept | Your ch07 hand-built equivalent |
|---|---|
| `Agent(role, goal, backstory)` | a specialist: `run_agent` with a scoped toolset and a role system prompt |
| `Agent.tools=[...]` (a subset) | least-privilege scoping — the checker can't move money it never holds |
| `Task(description, expected_output)` | the focused question you handed a specialist, plus its output contract |
| `context=[triage_task]` | `forward()` — passing one agent's messages to the next |
| `Crew(process=Process.sequential)` | the supervisor loop that orders the work and threads results |
| `Process.hierarchical` (below) | a supervisor delegating to specialists and reading results back |
| `crew.kickoff()` | calling `run_agent` on the whole team and collecting the answer |

## The other process: a manager delegates

`Process.sequential` fixes the order up front. `Process.hierarchical` does not: you give the crew a `manager_llm` and CrewAI spins up an extra **manager agent** that decides which worker handles what, delegates, and assembles the result — ch07's supervisor-with-agents-as-tools, auto-generated. You don't assign agents to tasks; the manager does. It is strictly more machinery — an extra model in the loop making delegation calls — so it costs more per ticket. We construct one to show its shape rather than spend the tokens on a full run.

In [ ]:
manager_crew = Crew(
    agents=[triager, checker],
    tasks=[Task(description="Triage TKT-2205 by delegating the lookups, then decide.",
                expected_output="One line: '<decision> | <policy_id> | $<amount>'.")],
    process=Process.hierarchical, manager_llm=llm, verbose=False)

print("process:", manager_crew.process)
print("manager_llm wired:", manager_crew.manager_llm is not None)
print("tasks pre-assigned to an agent?", manager_crew.tasks[0].agent is not None,
      "(the manager assigns at kickoff)")

> **What you should see:** the crew reports `Process.hierarchical`, a wired `manager_llm`, and a task with **no** agent pre-assigned — the manager will pick one at `kickoff()`. That auto-created manager is exactly the ch07 supervisor: it holds the workers as delegate-able tools and calls them. Handy when the division of labour is genuinely dynamic; wasteful when, as here, two fixed roles in sequence already do the job.

## Machinery map: CrewAI feature to the part you built

The Part 5 through-line: every framework feature is a piece of machinery you already built. CrewAI's is a clean one-to-one with the loop, tools, and multi-agent machinery you built by hand.

| CrewAI concept | Your hand-built equivalent | Built in |
|---|---|---|
| `LLM(model="openrouter/...")` | `shoplab.llm.complete` over LiteLLM — the model boundary | ch01 |
| `@tool` decorator | `shoplab.tools.Tool` — a fn plus a JSON schema the model can call | ch02 |
| `Agent` (role + scoped tools) | a specialist: `run_agent(system=..., toolset=scoped(...))` | ch07 |
| `Task` + `expected_output` | the rendered question + the `finish` output contract | ch01/07 |
| `context=[...]` handoff | `forward()` passing the conversation on | ch07 |
| `Crew(Process.sequential)` | the supervisor ordering work and threading results | ch07 |
| `Process.hierarchical` + `manager_llm` | the supervisor calling agents-as-tools | ch07 |
| `crew.kickoff()` + `token_usage` | `run_agent` + the `LEDGER`'s per-call token/cost rows | ch02/08 |

## When the team metaphor helps, and when it is a tax

CrewAI makes multi-agent systems *feel* effortless: name some roles, write some goals, pick a process, and you have a crew. That ease is exactly the trap ch07 warned about. The org-chart metaphor is so natural that it invites you to add roles an org chart would have — a "researcher", a "writer", a "reviewer" — whether or not the task needs the split. Each role you add is another model filling its own context, another boundary a finding crosses and degrades at, and more tokens and latency for the same ticket. On our run the second agent *earned* its place, because it held a policy tool the first did not; that is the test.

So use the same rule you brought from ch07. Reach for a second `Agent` when a subtask needs **isolation** or a **different toolset**, and measure whether the crew actually beats one well-equipped agent — cheaper-but-wrong is the most expensive outcome on a desk that moves money. CrewAI's gift is that the roles, tasks, delegation, and result-threading are written and tested for you. Its bill is the one every framework charges: the loop is now CrewAI's opinion, not yours, and the tidy org chart is easy to over-staff.

| Reach for a crew when | Stay with one agent when |
|---|---|
| subtasks need different, scoped toolsets (our triager vs checker) | one toolset and one context handle the whole job |
| a role must be walled off from risky tools | the split only makes the diagram look tidy |
| the division of labour is dynamic (hierarchical) | a fixed sequence of steps is all you have — a workflow, not a team |

## Recap

| Concept | One-liner |
|---|---|
| Team archetype | a `Crew` of role-playing `Agent`s working `Task`s under a `Process` — ch07's specialists, supervisor, and handoffs, packaged. |
| Own venv + kernel | `crewai` pins `mcp~=1.28.1` and `pydantic<2.13`, both below what the main stack resolved — a real conflict, not tidiness. |
| `LLM` = LiteLLM string | a CrewAI model is the same `openrouter/deepseek` router string from the ch01 model boundary. |
| `@tool` | a docstringed function becomes a callable tool — `shoplab.tools.Tool` by another name. |
| `Agent` = specialist | a `role`/`goal`/`backstory` plus a scoped toolset is your ch07 `run_agent` specialist. |
| `context=[task]` | forwards one task's output to the next — ch07's `forward()` handoff. |
| Sequential vs hierarchical | fixed order, or an auto-created manager agent that delegates — ch07's supervisor. |
| The tax | the org-chart metaphor invites over-staffing; a second agent must buy isolation or a toolset, not tidiness. |

## Exercises

1. **Add a third role.** Give the crew an `arithmetic` agent scoped to a single `calc` tool (wrap `shoplab`'s calculator as a `@tool`), and insert its task between the triager and the checker so the amount is *computed*, not guessed. Does pinning the math to a dedicated role stop the triager's dollar figure from wobbling — and what did the extra agent cost in requests (read `result.token_usage`)?
2. **Switch to hierarchical.** Take the two agents and run them under `Process.hierarchical` with `manager_llm=llm` for real. Compare `token_usage.successful_requests` against the sequential run: how many extra calls did the manager's delegation add, and did the final decision change? (Cost note: still cents, but several times the sequential run.)
3. **Name the packaging.** For each of `Agent`, `Task`, `context=[...]`, and `Process.hierarchical`, write one sentence identifying the exact ch07 idea it packages and one thing CrewAI adds on top (a default, an integration, or a convenience). Where does the mapping stop being one-to-one?

**Next up:** ch19 — the capstone. We stop meeting frameworks one at a time and bring the whole course together: the hand-built machinery, the protocols (MCP, A2A), and the framework archetypes, assembled into one production-shaped Larkspur agent.